## Resampling Methods

### Validation Set Approach

In [1]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
summarize,
poly)
from sklearn.model_selection import train_test_split

In [2]:
from functools import partial
from sklearn.model_selection import \
(cross_validate,
KFold,
ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

The Auto dataset is loaded using the load_data('Auto') function, which contains various attributes of automobiles, such as horsepower, weight, and fuel efficiency. To facilitate model training and evaluation, the dataset is split into training and validation sets using the train_test_split function from sklearn.model_selection. The validation set consists of 196 observations, as specified by test_size=196, while the remaining data is used for training. Additionally, random_state=0 ensures reproducibility by fixing the random seed. This setup allows us to train models on one portion of the data while evaluating their performance on a separate, unseen validation set.

In [3]:
Auto = load_data('Auto')
Auto_train, Auto_valid = train_test_split(Auto,
test_size=196,
random_state=0)

In this step, a simple linear regression model is built to examine the relationship between horsepower and miles per gallon (mpg). The variable hp_mm is defined using ModelSpec (MS), selecting horsepower as the predictor. The fit_transform method is applied to the training dataset (Auto_train), creating X_train, which represents the transformed horsepower feature. The response variable y_train is extracted from the dataset using the mpg column. An Ordinary Least Squares (OLS) regression model is then specified using statsmodels.api (sm.OLS) with y_train as the dependent variable and X_train as the independent variable. Finally, the model is fitted using the fit() method, producing regression results that can be analyzed further.

In [4]:
hp_mm = MS(['horsepower'])
X_train = hp_mm.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train, X_train)
results = model.fit()

To evaluate the performance of the linear regression model, the validation dataset (Auto_valid) is transformed using the same model specification (hp_mm.transform), creating X_valid, which represents the transformed horsepower feature. The actual miles per gallon (mpg) values for the validation set are stored in y_valid. Using the previously trained regression model (results), predictions are generated for X_valid and stored in valid_pred. The model’s accuracy is assessed by calculating the Mean Squared Error (MSE), which is obtained as the mean of the squared differences between the actual and predicted mpg values. The computed MSE value is 23.62, indicating the average squared error in the model's predictions.

In [5]:
X_valid = hp_mm.transform(Auto_valid)
y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)
np.mean((y_valid - valid_pred)**2)

23.61661706966988

The function evalMSE is defined to compute the Mean Squared Error (MSE) for different polynomial transformations of the horsepower variable in predicting miles per gallon (mpg). The function takes four arguments: terms (the predictor specification), response (the target variable), train (training dataset), and test (validation dataset). It applies a polynomial transformation to the horsepower feature, fits an Ordinary Least Squares (OLS) regression model using statsmodels, and computes the MSE by evaluating the squared differences between actual and predicted values in the validation set.

To assess model performance, the MSE array is initialized and filled by iterating through polynomial degrees 1 to 3, applying the evalMSE function to estimate validation errors for different polynomial models. The results show that as polynomial degree increases, the MSE generally decreases, indicating improved model fit. Additionally, the training and validation sets are re-split using a different random_state, leading to slightly different MSE values, reflecting the variability in model performance based on the dataset split.

In [8]:
def evalMSE(terms,
response,
train,
test):
    mm = MS(terms)
    X_train = mm.fit_transform(train)
    y_train = train[response]
    X_test = mm.transform(test)
    y_test = test[response]
    results = sm.OLS(y_train, X_train).fit()
    test_pred = results.predict(X_test)
    return np.mean((y_test - test_pred)**2)

In [9]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],'mpg',Auto_train,Auto_valid)
    
MSE

array([23.61661707, 18.76303135, 18.79694163])

In [10]:
Auto_train, Auto_valid = train_test_split(Auto, test_size=196, random_state=3)
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],'mpg',Auto_train,Auto_valid)
MSE

array([20.75540796, 16.94510676, 16.97437833])

### Cross Validation

In [11]:
hp_model = sklearn_sm(sm.OLS,
MS(['horsepower']))
X, Y = Auto.drop(columns=['mpg']), Auto['mpg']
cv_results = cross_validate(hp_model,
X,
Y,
cv=Auto.shape[0])
cv_err = np.mean(cv_results['test_score'])
cv_err

24.23151351792924

To assess the performance of the linear regression model using cross-validation, the Ordinary Least Squares (OLS) regression is wrapped within sklearn_sm, allowing it to be compatible with scikit-learn's cross-validation framework. The dataset is split into predictors (X), which includes all columns except mpg, and the response variable (Y), which is mpg. The cross_validate function is then applied to perform leave-one-out cross-validation (LOOCV), where the number of folds is set to the total number of observations (Auto.shape[0]). The mean test error is computed from the cross-validation results and stored in cv_err, yielding a final cross-validation error of 24.23. This approach ensures that the model's performance is evaluated robustly across different data splits, reducing the risk of overfitting.

In [13]:
cv_error = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M,X,Y,cv=Auto.shape[0])
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

array([24.23151352, 19.24821312, 19.33498406, 19.42443033, 19.03323827])

In [14]:
A = np.array([3, 5, 9])
B = np.array([2, 4])
np.add.outer(A, B)

array([[ 5,  7],
       [ 7,  9],
       [11, 13]])

This section performs Leave-One-Out Cross-Validation (LOOCV) to evaluate polynomial regression models of increasing complexity. The variable H stores the horsepower values from the dataset as a NumPy array, while M represents the Ordinary Least Squares (OLS) regression model wrapped for compatibility with scikit-learn. A loop iterates over polynomial degrees from 1 to 5, generating polynomial features using np.power.outer, which constructs a Vandermonde-like matrix for polynomial expansion. The cross_validate function then computes the cross-validation error for each polynomial degree, with the number of folds set to the total number of observations (Auto.shape[0]). The mean cross-validation error (cv_error) is recorded for each polynomial model, showing a decreasing trend as polynomial complexity increases, indicating potential improvements in model fit.

In [19]:
cv_error = np.zeros(5)  # Initialize an array to store cross-validation errors
cv = KFold(n_splits=10, shuffle=True, random_state=0)  # Define 10-fold cross-validation

for i, d in enumerate(range(1, 6)):  # Loop over polynomial degrees from 1 to 5
    X = np.power.outer(H, np.arange(d + 1))  # Generate polynomial features
    M_CV = cross_validate(M, X, Y, cv=cv)  # Perform cross-validation
    cv_error[i] = np.mean(M_CV['test_score'])  # Compute mean test score

cv_error  # Display the cross-validation errors


array([24.20766449, 19.18533142, 19.27626666, 19.47848403, 19.13720581])

This section applies 10-fold Cross-Validation (CV) to polynomial regression models with increasing degrees. The array cv_error is initialized to store the cross-validation errors for polynomial degrees ranging from 1 to 5. The K-Fold Cross-Validation method (KFold) is set with n_splits=10, ensuring the dataset is split into 10 folds while maintaining a shuffled and reproducible split (random_state=0). For each polynomial degree, the predictor variable horsepower (H) is transformed using np.power.outer, which generates polynomial features up to the specified degree. The cross-validation process is carried out using cross_validate, and the mean test error is computed for each polynomial model. The results indicate a decreasing cross-validation error as the polynomial degree increases, suggesting that higher-degree polynomial models may provide a better fit to the data.

### Bootstrap

In [20]:
Portfolio = load_data('Portfolio')
def alpha_func(D, idx):
    cov_ = np.cov(D[['X','Y']].loc[idx], rowvar=False)
    return ((cov_[1,1] - cov_[0,1]) /
(cov_[0,0]+cov_[1,1]-2*cov_[0,1]))

In [21]:
alpha_func(Portfolio, range(100))

0.57583207459283

In [22]:
rng = np.random.default_rng(0)
alpha_func(Portfolio,
rng.choice(100,
100,
replace=True))

0.6074452469619004

This section applies bootstrap resampling to estimate the variability of the portfolio allocation statistic computed by the alpha_func function. Initially, alpha_func(Portfolio, range(100)) computes the allocation statistic using all 100 observations in the dataset, yielding a value of 0.5758.

To introduce resampling, a random number generator (rng) is initialized using np.random.default_rng(0) to ensure reproducibility. The dataset is then resampled using rng.choice(100, 100, replace=True), which randomly selects 100 indices with replacement. This resampled dataset is passed to alpha_func, producing a slightly different estimate (0.6074), reflecting the effect of sampling variability.

Bootstrap resampling like this helps quantify the uncertainty in portfolio allocation decisions by generating multiple resampled estimates and constructing confidence intervals.

In [23]:
def boot_SE(func,D,n=None,B=1000,seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index,n,replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

In [24]:
alpha_SE = boot_SE(alpha_func,Portfolio,B=1000,seed=0)
alpha_SE

0.09118176521277699